In [1]:
import os
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_current_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups
from lvt.philadelphia import (
    tax_year_params, parcel_cache_path, split_zero_building_parcels,
    compute_lycd_land_values,
)

CITY_NAME = 'philadelphia'
STATE_FIPS = '42'
COUNTY_FIPS = '101'
LAND_IMPROVEMENT_RATIO = 4.0

# --- Tax year -------------------------------------------------------------------
# Rates and the revenue-validation target live in lvt/philadelphia.py, keyed by year and
# cited there. Do NOT hardcode a millage here: the combined rate has been 1.3998% for
# years, but the City/School split moved at TY2025, which silently invalidates the
# city-only cross-check without changing anything the model computes.
TAX_YEAR = int(os.environ.get('LVT_TAX_YEAR', 2026))   # override: LVT_TAX_YEAR=2027
TY = tax_year_params(TAX_YEAR)
MILLAGE = TY.combined_mills
PARCEL_PATH = parcel_cache_path(TAX_YEAR)
MODEL_TYPE = f'split_rate_4to1_lycd_ty{TAX_YEAR}'
EXPORT_SUFFIX = f'_lycd_ty{TAX_YEAR}'

print(TY.describe())
print(f'source: {TY.source}')

# GMA zone assignment: static reference extracted from OPA's 2025 GMA PDF
# (parcel centroid → L1/L2/L3 zone labels; 17 / 84 / 613 zones)
GMA_PATH = Path('data/parcel_gma_assignment.parquet')

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

TY2026: 0.6159% city + 0.7839% school = 1.3998% (13.998 mills) | city target $891,102,000 (projection) | homestead $100,000
source: Rate: City of Philadelphia Quarterly City Managers Report, period ending 2026-03-31, Summary Table R-1 ('FY 2026 Tax Rate: .6159% City plus .7839% School District Total 1.3998%'). Target: same report, Table R-2, Real Property 'Current' full-year Current Projection ($891,102k). This is a Q3 projection, not a closed-out actual.


C:\Users\druss\miniconda3\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


## Step 1: Load parcel data

In [2]:
if not PARCEL_PATH.exists():
    raise FileNotFoundError(
        f'{PARCEL_PATH} not found. Build it with:\n'
        f'    python scripts/build_philadelphia_parcel_cache.py --year {TAX_YEAR}\n'
        'The cache is keyed by tax year on purpose — opa_properties_public always carries '
        'the latest assessment year, so an unsuffixed cache makes it easy to model one '
        "year's taxable values against another year's expectations with no visible symptom."
    )
gdf = gpd.read_parquet(PARCEL_PATH)
_required = {'parcel_number', 'taxable_land', 'taxable_building', 'market_value',
             'exempt_land', 'exempt_building', 'pin', 'category_code', 'total_area'}
_missing = _required - set(gdf.columns)
if _missing:
    raise ValueError(
        f'{PARCEL_PATH} is missing columns {sorted(_missing)} — rebuild it with '
        f'scripts/build_philadelphia_parcel_cache.py --year {TAX_YEAR} --force'
    )
gdf['parcel_number'] = gdf['parcel_number'].astype(str).str.zfill(9)
print(f'Loaded {len(gdf):,} parcels for TY{TAX_YEAR}')
print(f'  taxable base: ${(gdf["taxable_land"].sum() + gdf["taxable_building"].sum())/1e9:.3f}B')

Loaded 583,249 parcels for TY2026
  taxable base: $152.997B


## Step 2: Compute LYCD land values (shared construction)

Land value: GMA-hierarchical LYCD, `zone_psf x land_pct x lot_area`, built by
`lvt.philadelphia.compute_lycd_land_values` — the single implementation every
Philadelphia LYCD notebook now shares (this notebook and
`model_lycd_reassessment.ipynb`), so the land surface cannot silently diverge between
them. See the function's docstring for the exact algorithm; in summary:

1. **Lot area, one convention, no spatial join.** OPA `total_area` first (~94.5% of
   parcels), then a Mercator-corrected DOR PIN-polygon area, then KNN from parcels
   already on that convention. The method is exactly scale-invariant in area — a
   uniform area error cancels completely — so only a *mixed*-convention error moves
   results. An earlier version of this notebook mixed a true-ground-area source with a
   Web-Mercator-inflated one, tripling the city's apparent land area; the function
   guards against that by raising if the total exceeds 1.5x Philadelphia's actual land
   area.
2. **Zone rate.** `median(market_value / lot_area)` over *improved* parcels, at the
   finest OPA Geographic Market Area level with at least 50 improved parcels
   (L3, else L2, else L1, of 613/84/17 zones).
3. **Land allocation.** 20% of the zone rate for improved parcels (OPA's own default),
   100% for vacant ones — OPA under-assesses bare land, and the allocation preserves
   that development-potential signal rather than suppressing it.
4. **KNN fallback** for the parcels outside GMA coverage, imputing a neighbour's
   *dollar* land value directly rather than a zone rate applied to the parcel's own
   area — a known limitation (audit 2026-08-08 finding 5).
5. **Market-value cap**, improved parcels only: land value cannot exceed the parcel's
   own market value. Vacant parcels are exempt from the cap for the same reason as (3).

`docs/LYCD_LAND_MODEL_ROADMAP.md` records what this method is (the *allocation*
technique, least preferred in the IAAO hierarchy), why each of these choices was made,
and what an assessor-grade replacement would need.

In [3]:
PIN_AREA_PATH = DATA_DIR / 'parcel_areas_by_pin_current.parquet'

if not PIN_AREA_PATH.exists():
    raise FileNotFoundError(
        f'{PIN_AREA_PATH} not found. Build it with:\n'
        '    python scripts/fetch_dor_parcel_areas.py\n'
        'Do NOT fall back to parcel_areas_by_pin.parquet — those areas are Web Mercator '
        '(inflated ~1.704x at this latitude) and mixing them with OPA total_area puts ~5% of '
        'parcels on a different area scale. See the Step 2 notes above.'
    )

pin_areas = pd.read_parquet(PIN_AREA_PATH)
pin_areas['pin'] = pin_areas['pin'].astype(str).str.strip()
print(f'PIN-area lookup: {len(pin_areas):,} PINs (true ground sqft, Mercator-corrected)')
print(f'  median lot: {pin_areas["pin_area_sqft"].median():,.0f} sqft')

PIN-area lookup: 580,097 PINs (true ground sqft, Mercator-corrected)
  median lot: 1,365 sqft


In [4]:
if not GMA_PATH.exists():
    raise FileNotFoundError(
        f'{GMA_PATH} not found. This is the static GMA zone-assignment reference file; '
        'see model_lycd_reassessment.ipynb Step 2 or CLAUDE.md for how it was built.'
    )
gma = pd.read_parquet(GMA_PATH)

lycd = compute_lycd_land_values(gdf, gma, pin_areas)
gdf = lycd.gdf
print(lycd.describe())
print(f"  lot area KNN-imputed: {lycd.diagnostics['n_area_knn']:,} parcels")
print(f"  land value KNN-imputed (no GMA zone): {lycd.diagnostics['n_land_knn']:,} parcels "
      "-- a spatial smooth of a neighbour's dollar land value, not this parcel's own zone "
      "rate x area; see docs/LYCD_LAND_MODEL_ROADMAP.md.")

Lot area source: opa_total_area=550,807, knn=30,696, pin_dor=1,351, pin_override=395
  OPA records overridden by surveyed polygon: 395
  total lot area = 0.84x the city (expect <1.0)
GMA assignment: 527,364 matched (90.4%); L3=526,329, knn=55,885, L2=929, L1=106
Market-value cap (improved only): 21,727 parcels, $106.41B -> $82.37B (22.6% removed)
  lot area KNN-imputed: 30,696 parcels
  land value KNN-imputed (no GMA zone): 55,885 parcels -- a spatial smooth of a neighbour's dollar land value, not this parcel's own zone rate x area; see docs/LYCD_LAND_MODEL_ROADMAP.md.


## Step 3: Summarize LYCD land base

In [5]:
total_lycd_land = gdf['lycd_land_value'].sum()
total_opa_land  = gdf['taxable_land'].sum()
print(f'Total LYCD land base:    ${total_lycd_land/1e9:.2f}B')
print(f'Total OPA taxable land:  ${total_opa_land/1e9:.2f}B')
print(f'Ratio LYCD/OPA:          {total_lycd_land/total_opa_land:.2f}x')
print()
print('LYCD land value by GMA level (median $):')
print(gdf.groupby('gma_level')['lycd_land_value'].median().sort_values(ascending=False).to_string())
print()
print('LYCD land value percentiles (all parcels):')
for p in [10, 25, 50, 75, 90, 99]:
    v = gdf['lycd_land_value'].quantile(p/100)
    print(f'  p{p:2d}: ${v:,.0f}')


Total LYCD land base:    $82.37B
Total OPA taxable land:  $43.00B
Ratio LYCD/OPA:          1.92x

LYCD land value by GMA level (median $):
gma_level
L2     1.045000e+06
L1     6.270000e+05
knn    1.261000e+05
L3     4.593303e+04

LYCD land value percentiles (all parcels):
  p10: $20,648
  p25: $30,849
  p50: $49,078
  p75: $85,440
  p90: $173,246
  p99: $1,108,481


## Step 4: Categorize parcels (same overrides as OPA model)

In [6]:
gdf['category_code'] = (
    pd.to_numeric(gdf['category_code'], errors='coerce')
    .astype('Int64')
    .astype(str)
)

CATEGORY_MAP = {
    '1':  'Single Family Residential',
    '2':  'Small Multi-Family (2-4 units)',
    '3':  'Mixed Use',
    '4':  'Commercial',
    '5':  'Industrial',
    '6':  'Vacant Land',
    '7':  'Other Commercial',
    '8':  'Other Residential',
    '9':  'Hotel',
    '10': 'Office / Commercial Condo',
    '11': 'Other',
    '12': 'Vacant Land',
    '13': 'Vacant Land',
    '14': 'Large Multi-Family (5+ units)',
    '15': 'Retail / General Commercial',
}
gdf['PROPERTY_CATEGORY'] = gdf['category_code'].map(CATEGORY_MAP).fillna('Other')

# Override 1: $0 improvement -> Vacant Land
gdf.loc[gdf['taxable_building'] <= 0, 'PROPERTY_CATEGORY'] = 'Vacant Land'

# Override 2: a $0 taxable building line has three different causes, and calling all of
# them "abated" put ~13K homesteaded rowhomes in the abated bucket -- then revoked their
# Homestead Exemption under the reform. Split on the year's statutory homestead cap.
GENUINE_VACANT_CODES = {'6', '12', '13'}
_zb = split_zero_building_parcels(
    gdf, gdf['PROPERTY_CATEGORY'], TY.homestead_exemption, CATEGORY_MAP,
    genuine_vacant_codes=tuple(GENUINE_VACANT_CODES),
)
gdf['PROPERTY_CATEGORY'] = _zb.category
abated_mask = _zb.abated
print(_zb.describe())

# Override 3: OPA-vacant with nonzero building value
improved_vacant_mask = (
    gdf['category_code'].isin(GENUINE_VACANT_CODES) &
    (gdf['taxable_building'] > 0)
)
gdf.loc[improved_vacant_mask, 'PROPERTY_CATEGORY'] = 'Improved Vacant Land'

gdf['taxable_total'] = (gdf['taxable_land'] + gdf['taxable_building']).clip(lower=0)
gdf['full_exmp'] = (gdf['taxable_total'] <= 0).astype(int)

# Override 4: fully exempt parcels
EXEMPT_CATEGORY_MAP = {k: v + ' — Exempt' for k, v in CATEGORY_MAP.items()}
exempt_mask = gdf['full_exmp'] == 1
gdf.loc[exempt_mask, 'PROPERTY_CATEGORY'] = (
    gdf.loc[exempt_mask, 'category_code']
    .map(EXEMPT_CATEGORY_MAP)
    .fillna('Other — Exempt')
)

print(f'Total parcels: {len(gdf):,}')
print(f'Fully exempt: {gdf["full_exmp"].sum():,}  |  '
      f'Abated: {abated_mask.sum():,}  |  '
      f'Improved vacant: {improved_vacant_mask.sum():,}  |  '
      f'Taxable: {(gdf["full_exmp"] == 0).sum():,}')
print()
print('Property category distribution:')
print(gdf['PROPERTY_CATEGORY'].value_counts().to_string())

zero-building line: 14,287 abated | 13,995 homestead-zeroed (96.3% confirmed by OPA's homestead flag) | 1,119 genuinely $0 improvement


Total parcels: 583,249
Fully exempt: 36,932  |  Abated: 14,287  |  Improved vacant: 880  |  Taxable: 546,317

Property category distribution:
PROPERTY_CATEGORY
Single Family Residential                  430570
Small Multi-Family (2-4 units)              38685
Vacant Land                                 30557
Single Family Residential — Exempt          20078
Abated / Construction Exemption             14287
Mixed Use                                   13743
Vacant Land — Exempt                        11714
Commercial                                   8802
Industrial                                   3553
Commercial — Exempt                          3298
Large Multi-Family (5+ units)                3002
Other Residential                            1108
Small Multi-Family (2-4 units) — Exempt      1026
Improved Vacant Land                          880
Office / Commercial Condo                     825
Large Multi-Family (5+ units) — Exempt        264
Industrial — Exempt                     

## Step 5: Current tax (OPA taxable values — revenue baseline)

In [7]:
gdf['millage_rate'] = MILLAGE

current_revenue, _, gdf = calculate_current_tax(
    df=gdf,
    tax_value_col='taxable_total',
    millage_rate_col='millage_rate',
    exemption_flag_col='full_exmp',
)

city_revenue = gdf['taxable_total'].mul(TY.city_mills / 1000).sum()

print(f'Modeled combined levy (city + school):  ${current_revenue:,.0f}')
print(f'Implied city-only portion ({TY.city_rate_pct}%):   ${city_revenue:,.0f}')

if TY.city_revenue_target is None:
    # TY2027: bills are not due until March 2027, so there are no collections to check
    # against. This run is a forward-looking scenario, not a validated baseline.
    print(f'\nNO REVENUE VALIDATION for TY{TAX_YEAR}.')
    print(f'  {TY.source}')
else:
    gap_pct = (city_revenue / TY.city_revenue_target - 1) * 100
    print(f'City-only target ({TY.target_kind}):            ${TY.city_revenue_target:,}')
    print(f'City portion gap: {gap_pct:+.2f}%  (expected: a few % over, from delinquency)')
    assert abs(gap_pct) < 10.0, (
        f'City gap {gap_pct:.2f}% exceeds 10% for TY{TAX_YEAR}. Check that the assessment '
        f'year, the City rate ({TY.city_rate_pct}%) and the revenue target all refer to the '
        'same billing year — see lvt/philadelphia.py.'
    )

Modeled combined levy (city + school):  $2,141,653,043
Implied city-only portion (0.6159%):   $942,308,979
City-only target (projection):            $891,102,000
City portion gap: +5.75%  (expected: a few % over, from delinquency)


## Step 6: Build GMA LYCD reform base

Land value: GMA hierarchical LYCD (`lycd_land_value`).
Building value: OPA `taxable_building` for non-abated parcels (post-exemption, preserves
Homestead and other reliefs).

For abated parcels (OPA shows zero taxable_building due to active 10-year construction
abatements): use OPA's own recorded assessed building value, `exempt_building` — the
value the abatement is shielding from tax, not an estimate of it — with a
`market_value - taxable_land` fallback for the ~2,100 mid-construction parcels where
`exempt_building = 0`. This is the same restoration `model.ipynb` and
`model_lycd_post_abatement.ipynb` use, so the reform scenario differs from the OPA panel
by land values alone rather than also by how the building value is estimated.

(An earlier version of this cell used `model_building = 4 x model_land` instead, on the
reasoning that `model.ipynb` did the same and that using `exempt_building` here would
make this scenario match the *post-abatement* baseline. Both premises were wrong:
`exempt_building` is not a post-abatement treatment -- `current_tax` stays land-only
either way, and only the reform-scenario `model_building` changes -- and `model.ipynb`
has since moved to `exempt_building` itself, so the 4x version left this notebook
differing from the OPA panel by imputation method as well as by land source. See the
reconciled note in `CLAUDE.md`'s Philadelphia section.)


In [8]:
gdf['model_land']     = gdf['lycd_land_value'].clip(lower=0)
gdf['model_building'] = pd.to_numeric(gdf['taxable_building'], errors='coerce').fillna(0).clip(lower=0)

abated = gdf['PROPERTY_CATEGORY'] == 'Abated / Construction Exemption'

# OPA's own recorded assessed building value (the value the abatement shields from tax),
# with a market_value - taxable_land fallback for mid-construction parcels where
# exempt_building = 0. Same restoration as model.ipynb and model_lycd_post_abatement.ipynb.
exempt_bldg  = pd.to_numeric(gdf['exempt_building'], errors='coerce').fillna(0)
market_val   = pd.to_numeric(gdf['market_value'],    errors='coerce').fillna(0)
tax_land     = pd.to_numeric(gdf['taxable_land'],     errors='coerce').fillna(0)
implied_bldg = (market_val - tax_land).clip(lower=0)
abated_bldg  = exempt_bldg.where(exempt_bldg > 0, implied_bldg)
gdf.loc[abated, 'model_building'] = abated_bldg[abated].values

n_exempt_bldg = int((abated & (exempt_bldg > 0)).sum())
n_fallback    = int((abated & (exempt_bldg <= 0)).sum())
print(f'Abated parcels using exempt_building:        {n_exempt_bldg:,}')
print(f'Abated parcels using market_value fallback:  {n_fallback:,}')
print()
print(f'Reform land base:          ${gdf["model_land"].sum()/1e9:.2f}B')
print(f'Reform improvement base:   ${gdf["model_building"].sum()/1e9:.2f}B')
print(f'  of which abated bldg:    ${gdf.loc[abated,"model_building"].sum()/1e9:.2f}B')
print(f'OPA taxable land base:     ${pd.to_numeric(gdf["taxable_land"],errors="coerce").sum()/1e9:.2f}B')
print(f'OPA taxable building base: ${pd.to_numeric(gdf["taxable_building"],errors="coerce").sum()/1e9:.2f}B')


Abated parcels using exempt_building:        14,269
Abated parcels using market_value fallback:  18

Reform land base:          $82.37B
Reform improvement base:   $126.21B
  of which abated bldg:    $16.21B
OPA taxable land base:     $43.00B
OPA taxable building base: $110.00B


## Step 7: Revenue-neutral split-rate model (4:1 land:improvement)

In [9]:
taxable = gdf[gdf['full_exmp'] == 0].copy()

land_millage, improvement_millage, new_revenue, taxable = model_split_rate_tax(
    df=taxable,
    land_value_col='model_land',
    improvement_value_col='model_building',
    current_revenue=taxable['current_tax'].sum(),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

# Recombine exempt parcels
exempt = gdf[gdf['full_exmp'] == 1].copy()
exempt['new_tax'] = 0.0
exempt['tax_change'] = 0.0
exempt['tax_change_pct'] = 0.0
exempt['taxable_land_value'] = 0.0
exempt['taxable_improvement_value'] = 0.0
gdf = pd.concat([taxable, exempt]).sort_index()

print(f'Land millage:        {land_millage:.4f} mills')
print(f'Improvement millage: {improvement_millage:.4f} mills')
print(f'Revenue check:       ${new_revenue:,.0f} (target: ${taxable["current_tax"].sum():,.0f})')
print()

category_summary = calculate_category_tax_summary(
    df=gdf,
    category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
)
print_category_tax_summary(category_summary, title='Philadelphia — 4:1 Split-Rate Tax Impact (LYCD Land Values)')

Land millage:        22.7603 mills
Improvement millage: 5.6901 mills
Revenue check:       $2,141,653,043 (target: $2,141,653,043)




Philadelphia — 4:1 Split-Rate Tax Impact (LYCD Land Values)
                               Category  Count Total Tax Δ ($) Total Δ (%) Mean Δ ($) Median Δ ($) Avg % Δ Median % Δ % Parcels > +10% % Parcels < -10%
              Single Family Residential 430570   $-193,581,262      -15.6%      $-450        $-498   12.4%     -24.5%            18.9%            68.3%
         Small Multi-Family (2-4 units)  38685    $-88,472,161      -35.0%    $-2,287      $-1,401  -28.6%     -35.4%             5.7%            88.1%
                            Vacant Land  30557    $292,823,593      560.1%     $9,583       $2,371 1117.6%     545.8%            94.9%             4.2%
     Single Family Residential — Exempt  20078              $0        0.0%         $0           $0    0.0%       0.0%             0.0%             0.0%
        Abated / Construction Exemption  14287    $103,997,615      282.7%     $7,279       $2,702  316.3%     188.6%            99.8%             0.1%
                           

## Step 8: Census join

In [10]:
import concurrent.futures

_fips = STATE_FIPS + COUNTY_FIPS
try:
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
        _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
        try:
            census_data, census_gdf = _future.result(timeout=90)
            gdf = match_to_census_blockgroups(gdf, census_gdf)
            if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
                gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
            if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
                gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
            print(f'Census join: {gdf["std_geoid"].notna().mean()*100:.1f}% matched')
        except concurrent.futures.TimeoutError:
            print('Census API timed out — skipping census join')
            for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
                gdf[_col] = float('nan')
except Exception as e:
    print(f'Census join failed: {e}')
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        gdf[_col] = float('nan')

Census join: 100.0% matched


## Step 9: Export and visualize

In [11]:
out_df = save_standard_export(
    df=gdf,
    city=f'{CITY_NAME}{EXPORT_SUFFIX}',
    output_path=f'../../analysis/data/{CITY_NAME}{EXPORT_SUFFIX}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
    parcel_id_col='parcel_number',
)

from lvt.viz import create_city_report
create_city_report(out_df, f'{CITY_NAME}{EXPORT_SUFFIX}', show=False)
print('Done.')

  [warn] philadelphia_lycd_ty2026: non-standard property categories (will be preserved): ['Abated / Construction Exemption', 'Commercial — Exempt', 'Hotel — Exempt', 'Improved Vacant Land', 'Industrial — Exempt', 'Large Multi-Family (5+ units) — Exempt', 'Mixed Use — Exempt', 'Office / Commercial Condo — Exempt', 'Other Commercial — Exempt', 'Other Residential — Exempt', 'Other — Exempt', 'Retail / General Commercial — Exempt', 'Single Family Residential — Exempt', 'Small Multi-Family (2-4 units) — Exempt', 'Vacant Land — Exempt']


  ✓ philadelphia_lycd_ty2026: 583,249 rows → ../../analysis/data/philadelphia_lycd_ty2026.csv  [model: split_rate_4to1_lycd_ty2026]


create_city_report: excluded 36,932 fully-exempt parcels (583,249 → 546,317 modeled).


Done.
